### **Import Packages and Libraries**

In [ ]:
import cv2
import os

### **Definition of frame extraction function**

In [ ]:
def extract_frames(video_path, num_frames, output_dir='frames/'):
    print("Starting frame extraction...")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Video file could not be opened.")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Total frames in video: {total_frames}")

    frame_interval = total_frames // num_frames

    count = 0

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if count % frame_interval == 0:
            frame_filename = os.path.join(output_dir, f'frame_{count}.jpg')
            cv2.imwrite(frame_filename, frame)

        count += 1

    cap.release()
    cv2.destroyAllWindows()

### **Using the function**

In [ ]:
extract_frames('Videos/handheld1.mp4', 50, 'frames/1')

In [ ]:
extract_frames('Videos/handheld6.mp4', 100, 'frames/2')

In [ ]:
extract_frames('Videos/handheld17_reflection_test.mp4', 100, 'frames/3')

In [ ]:
extract_frames('Videos/handheld4.mp4', 100, 'frames/4')

In [ ]:
extract_frames('Videos/handheld11.mp4', 100, 'frames/5')

In [ ]:
extract_frames('Videos/out2.mp4', 150, 'frames/6')

### **Annotating of frames**

In [ ]:
!pip install labelme

#### Note: Annotation was done with labelme for 600 frame.

### **Display a number of extracted frames**

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

folders = ['rectangle_annotation_frames/1', 'rectangle_annotation_frames/2', 'rectangle_annotation_frames/3', 'rectangle_annotation_frames/4', 'rectangle_annotation_frames/5', 'rectangle_annotation_frames/6']

num_frames = 4

plt.figure(figsize=(20, 12))

plt.suptitle("Display a number of extracted frames", fontsize=16)

for i, folder in enumerate(folders, 1):
    all_frames = os.listdir(folder)

    selected_frames = random.sample(all_frames, num_frames)

    for j, frame_file in enumerate(selected_frames):
        frame_path = os.path.join(folder, frame_file)
        image = cv2.imread(frame_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.subplot(len(folders), num_frames, (i-1)*num_frames + j + 1)
        plt.imshow(image)
        plt.axis('off')
        plt.title(f'F{i} - Img{j+1}', fontsize=12)

plt.tight_layout()
plt.show()

### **Rename files**

In [ ]:
import os

main_dir = "frames/"

folders = ["1", "2", "3", "4", "5", "6"]

def rename_files(folder, subfolder):
    folder_path = os.path.join(main_dir, folder, subfolder)
    files = os.listdir(folder_path)

    for file_name in files:
        ext = os.path.splitext(file_name)[1]
        new_name = f"{folder}_{file_name}"
        src = os.path.join(folder_path, file_name)
        dst = os.path.join(folder_path, new_name)
        os.rename(src, dst)

for folder in folders:
    rename_files(folder, "")

In [ ]:
for folder in folders:
    rename_files(folder, "annotation")

### **Convert json format**

In [ ]:
import os
import json

main_dir = "frames/"

folders = ["1", "2", "3", "4", "5", "6"]

def convert_json_to_yolo(json_file, output_path):
    with open(json_file, 'r') as f:
        data = json.load(f)

    image_width = data['imageWidth']
    image_height = data['imageHeight']

    with open(output_path, 'w') as out_file:
        for shape in data['shapes']:
            label = shape['label']
            points = shape['points']

            if shape['shape_type'] == 'circle':
                cx, cy = points[0]
                radius = points[1][0] - cx
                x_min = cx - radius
                x_max = cx + radius
                y_min = cy - radius
                y_max = cy + radius
            else:
                x_min = min(points[0][0], points[1][0])
                x_max = max(points[0][0], points[1][0])
                y_min = min(points[0][1], points[1][1])
                y_max = max(points[0][1], points[1][1])

            x_center = (x_min + x_max) / 2.0 / image_width
            y_center = (y_min + y_max) / 2.0 / image_height
            width = (x_max - x_min) / image_width
            height = (y_max - y_min) / image_height

            out_file.write(f"{label} {x_center} {y_center} {width} {height}\n")

In [ ]:
for folder in folders:
    json_folder = os.path.join(main_dir, folder, "annotation")
    txt_folder = os.path.join(main_dir, folder, "labels")
    os.makedirs(txt_folder, exist_ok=True)

    for json_file in os.listdir(json_folder):
        if json_file.endswith(".json"):
            json_path = os.path.join(json_folder, json_file)
            txt_filename = os.path.splitext(json_file)[0] + ".txt"
            txt_path = os.path.join(txt_folder, txt_filename)
            convert_json_to_yolo(json_path, txt_path)

### **Create dataset**

In [ ]:
import os
import shutil
import random

main_dir = "frames/"
output_dir = "dataset/"

os.makedirs(os.path.join(output_dir, "images/train"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "images/val"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "labels/val"), exist_ok=True)

folders = ["1", "2", "3", "4", "5", "6"]

split_ratio = 0.8

def copy_files(image_files, label_files, split):
    for i, file in enumerate(image_files):
        if i < split:
            shutil.copy(file, os.path.join(output_dir, "images/train"))
            shutil.copy(label_files[i], os.path.join(output_dir, "labels/train"))
        else:
            shutil.copy(file, os.path.join(output_dir, "images/val"))
            shutil.copy(label_files[i], os.path.join(output_dir, "labels/val"))

for folder in folders:
    image_dir = os.path.join(main_dir, folder)
    label_dir = os.path.join(main_dir, folder, "labels")

    images = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]
    labels = [os.path.join(label_dir, f.replace('.jpg', '.txt')) for f in os.listdir(image_dir) if f.endswith('.jpg')]

    labels = [f for f in labels if os.path.exists(f)]

    print(f"Folder: {folder}, Images found: {len(images)}, Labels found: {len(labels)}")

    if len(images) == 0 or len(labels) == 0:
        print(f"No valid images or labels found in folder: {folder}")
        continue

    combined = list(zip(images, labels))
    random.shuffle(combined)

    if len(combined) == 0:
        print(f"No pairs of images and labels found in folder: {folder}")
        continue

    images[:], labels[:] = zip(*combined)

    split = int(len(images) * split_ratio)

    copy_files(images, labels, split)

### **Training YOLO models**

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

YV5_model = YOLO("yolov5s.pt")

PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


In [ ]:
YV5_model.train(data="./dataset/data.yaml", epochs=100, batch=16, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.175 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.85  Python-3.12.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=yolov5s.pt, data=./dataset/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000019D81B95760>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [ ]:
pwd

'.'